In [ ]:
from transformers import pipeline
import requests
from openai import OpenAI
from transformers import BartForConditionalGeneration, BartTokenizer

device = "cuda"

API_key = "YOUR API_key"
CX_key = "YOUR CX_key"
DEEP_SEEK_API_key = "YOUR DEEP_SEEK_API_key"

In [13]:
question_ru = "Какие факторы способствует развитию нестандартного мышления и креативности у детей?"
#question_ru = "Do boys need to be vaccinated against HPV, and if so, at what age?"

1. Переводчик с русского на английский

In [14]:
translator_ru_en = pipeline("translation_ru_to_en", model="Helsinki-NLP/opus-mt-ru-en", device=0 if device == "cuda" else -1)

Device set to use cuda:0


In [15]:
def translate_question(question_ru):
    translated = translator_ru_en(question_ru)[0]["translation_text"]
    
    return translated

question_en = translate_question(question_ru)
print("Перевод на английский язык: ", question_en)

Перевод на английский язык:  What factors contribute to the development of non-standard thinking and creativity in children?


2. Генератор ключевых слов на основе дообученной модели

In [16]:
model = BartForConditionalGeneration.from_pretrained("./medical_keyword_extractor").to(device)
tokenizer = BartTokenizer.from_pretrained("./medical_keyword_extractor")

In [17]:
def extract_keywords(text, model, tokenizer, max_length=128):
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    output = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        num_beams=5,
        early_stopping=True
    )
    
    keywords = tokenizer.decode(output[0], skip_special_tokens=True)
    return keywords

keywords = extract_keywords(question_en, model, tokenizer)
print("Ключевые слова: ", keywords)

Ключевые слова:  Non-standard thinking, creativity, children


3. Поиск с помощью Google Programmable Search Engine по сгенерированным ключевым словам и извлечение трех первых ссылок из выдачи

In [ ]:
#API ключ и идентификатор поискового движка
api_key = API_key
cx = CX_key

search_terms = keywords.replace(", ", "+AND+")

url = f"https://www.googleapis.com/customsearch/v1?q={keywords}&key={api_key}&cx={cx}"

response = requests.get(url)

if response.status_code == 200:

    data = response.json()

    items = data.get('items', [])
    top_3_links = [item['link'] for item in items[:3]]

    print("Первые три ссылки:")
    for i, link in enumerate(top_3_links, 1):
        print(f"{i}. {link}")

else:
    print(f"Ошибка запроса: {response.status_code}")

Первые три ссылки:
1. https://www.nichd.nih.gov/sites/default/files/publications/pubs/nrp/documents/report.pdf
2. https://www.healthychildren.org/English/ages-stages/young-adult/Pages/What-Fuels-Perfectionism.aspx
3. https://www.nimh.nih.gov/health/publications/children-and-mental-health


4. Суммаризация и перевод содержания статей по отобранным ссылкам

In [ ]:
deep_seek_api_key = DEEP_SEEK_API_key

In [20]:
client = OpenAI(api_key=deep_seek_api_key, base_url="https://api.deepseek.com/v1")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": f"Кратко изложи основные идеи статьи по ссылкам {top_3_links}, сохраняя ключевые факты."\
         "Общий объем обзора по всем ссылкам: до 500 слов. Приведи ссылки, откуда ты взял соответствующую информацию."},
    ],
    stream=False
)

print(response.choices[0].message.content)

### Краткий обзор ключевых идей из статей  

1. **[National Reading Panel Report (NRP)](https://www.nichd.nih.gov/sites/default/files/publications/pubs/nrp/documents/report.pdf)**  
   - Основной фокус: эффективные методы обучения чтению.  
   - Ключевые выводы:  
     - **Фонетическое обучение** (осознание звуков в словах) критически важно для раннего чтения.  
     - **Систематическое обучение фонетике** улучшает навыки чтения у детей, особенно в начальной школе.  
     - **Беглость чтения** развивается через повторное чтение и контроль понимания текста.  
     - **Словарный запас** и **стратегии понимания** (например, вопросы по тексту) усиливают осмысленное чтение.  

2. **[What Fuels Perfectionism? (HealthyChildren.org)](https://www.healthychildren.org/English/ages-stages/young-adult/Pages/What-Fuels-Perfectionism.aspx)**  
   - Основные причины перфекционизма у подростков и молодых взрослых:  
     - **Давление общества и соцсетей**, где демонстрируются только "идеальные" результ